# Ignite-3B S10 - Shapley attribution + reward-hacking audit + plots

Consumes archive log from Cond C. Produces:
- Per-mutation Shapley credit (top-10)
- Reward-hacking rate per generation
- Sigmoid fit R_inf per condition
- BOCPD change-point posterior
- Gain-vs-generation power-law fit

Outputs: matplotlib PNGs + summary.json + README section.

In [ ]:
!pip install -q -U 'huggingface_hub' 'scipy' 'numpy' 'matplotlib' 'seaborn' kaggle
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')

In [ ]:
import subprocess, os
ARCHIVE_DATASET = 'vitorscrt/ignite-3b-cond-c-s05'
OUT = '/kaggle/working/analysis'
os.makedirs(OUT, exist_ok=True)
subprocess.run(['kaggle', 'datasets', 'download', '-d', ARCHIVE_DATASET, '-p', OUT, '--unzip', '--force'], check=True)

In [ ]:
from pathlib import Path
from eval.ignite.stats import shapley_attribution

log_path = Path(f'{OUT}/log.jsonl')
top = shapley_attribution(log_path, top_k=10)
print('Top-10 mutations by cumulative val_r contribution:')
for r in top:
    print(f"  {r['mutation_hash']}: {r['cumulative_contribution']:.4f}")

In [ ]:
import json
from pathlib import Path
log = [json.loads(l) for l in open(f'{OUT}/log.jsonl')]
cands = [r for r in log if r.get('type') == 'candidate' and r.get('val_r') is not None]
by_gen = {}
for c in cands:
    g = c['gen']
    dev, val = c.get('dev_r', 0), c.get('val_r', 0)
    hack = 1 if dev > val + 0.05 else 0   # dev > val by >5pp = hacking
    by_gen.setdefault(g, []).append(hack)
print('Reward-hacking rate per generation:')
for g in sorted(by_gen):
    v = by_gen[g]
    print(f'  gen{g}: {100*sum(v)/len(v):.1f}% ({sum(v)}/{len(v)})')

In [ ]:
import json, matplotlib.pyplot as plt
from eval.ignite.stats import sigmoid_fit, bocpd
gens = [r for r in log if r.get('type') == 'generation']
if gens:
    val_r = [g['val_r'] for g in gens]
    compute = [i+1 for i in range(len(val_r))]   # generations as compute proxy
    fit = sigmoid_fit(compute, val_r)
    bc = bocpd(val_r)
    print('Sigmoid fit:', json.dumps(fit, indent=2, default=str))
    print('BOCPD:', json.dumps(bc, indent=2))
    plt.plot(compute, val_r, 'o-')
    plt.xlabel('generation'); plt.ylabel('val reward')
    plt.title('Ignite-3B Cond C: val reward per generation')
    plt.savefig(f'{OUT}/gain_curve.png', dpi=150)
    plt.show()